### Dataset and Task Metadata

In [5]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="credit_card_clients_default",
    dataset_year="2009",
    domain_str="finance",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C55S3H",
    download_description="""
We download the data from the UCI repository and uzip it to a predefined folder.

mkdir -p local-data-warehouse/credit_card_clients_default/ && wget -P local-data-warehouse/credit_card_clients_default/ https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip && unzip local-data-warehouse/credit_card_clients_default/default+of+credit+card+clients.zip -d local-data-warehouse/credit_card_clients_default/ && rm local-data-warehouse/credit_card_clients_default/default+of+credit+card+clients.zip
""",
    # References
    academic_reference_bibtex="""@article{yeh2009comparisons,
  title={The comparisons of data mining techniques for the predictive accuracy of probability of default of credit card clients},
  author={Yeh, I-Cheng and Lien, Che-hui},
  journal={Expert systems with applications},
  volume={36},
  number={2},
  pages={2473--2480},
  year={2009},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="yeh2009comparisons",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
  - We rename the target variable and restore the original class names.
  - We drop the "ID" column.
  - Anomaly: the data has temporal features but the task is time-invariant.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="DefaultOnPaymentNextMonth",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="DefaultOnPaymentNextMonth",
)

## Preprocessing

In [6]:
import pandas as pd

df = pd.read_excel(dataset_mold.path / "default of credit card clients.xls", skiprows=[0])

df = df.drop(columns=["ID"])

target_feature = "DefaultOnPaymentNextMonth"
df = df.rename(columns={"default payment next month": target_feature})
df[target_feature] = df[target_feature].map({1: "Yes", 0: "No"}).astype("category")

cat_columns = ["SEX", "MARRIAGE", "EDUCATION"]
df[cat_columns] = df[cat_columns].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [7]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 30,000
Columns: 24
Use sampling: False (sample size: 30,000)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT6']
Rows remaining as candidates after top-10 filter: 1,248 (of 30,000)

#### Duplicate Report
Total duplicate rows: 35 (0.12% of dataset)
Duplicate rows ignoring target: 56 (0.19% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [8]:
# Sample Rows
df_head

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DefaultOnPaymentNextMonth
0,30000,1,2,2,25,0,0,0,0,0,0,8864,10062,11581,12580,13716,14828,1500,2000,1500,1500,1500,2000,No
1,150000,2,1,2,26,0,0,0,0,0,0,136736,125651,116684,101581,77741,77264,4486,4235,3161,2647,2669,2669,No
2,70000,2,3,1,32,0,0,0,0,0,0,70122,69080,68530,69753,70111,70212,2431,3112,3000,2438,2500,2554,No
3,130000,1,3,2,49,0,0,0,0,0,-1,20678,18956,16172,16898,11236,6944,1610,1808,7014,27,7011,4408,No
4,50000,2,2,2,36,0,0,0,0,0,2,94228,47635,42361,19574,20295,19439,2000,1500,1000,1800,0,1000,Yes


In [9]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,SEX,category,0.0,0.0,2.0,"2, 1"
1,EDUCATION,category,0.0,0.0,7.0,"2, 1, 3, 5, 4, 6, 0"
2,MARRIAGE,category,0.0,0.0,4.0,"2, 1, 3, 0"
3,DefaultOnPaymentNextMonth,category,0.0,0.0,2.0,"No, Yes"
4,LIMIT_BAL,int64,0.0,0.0,81.0,"50000, 20000, 30000, 80000, 200000, 150000, 100000, 180000, 360000, 60000"
5,AGE,int64,0.0,0.0,56.0,"29, 27, 28, 30, 26, 31, 25, 34, 32, 33"
6,PAY_0,int64,0.0,0.0,11.0,"0, -1, 1, -2, 2, 3, 4, 5, 8, 6"
7,PAY_2,int64,0.0,0.0,11.0,"0, -1, 2, -2, 3, 4, 1, 5, 7, 6"
8,PAY_3,int64,0.0,0.0,11.0,"0, -1, -2, 2, 3, 4, 7, 6, 5, 1"
9,PAY_4,int64,0.0,0.0,11.0,"0, -1, -2, 2, 3, 4, 7, 5, 6, 1"


In [10]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
LIMIT_BAL,30000.0,167484.322667,129747.661567,10000.0,1000000.0
AGE,30000.0,35.485500,9.217904,21.0,79.0
PAY_0,30000.0,-0.016700,1.123802,-2.0,8.0
PAY_2,30000.0,-0.133767,1.197186,-2.0,8.0
PAY_3,30000.0,-0.166200,1.196868,-2.0,8.0
PAY_4,30000.0,-0.220667,1.169139,-2.0,8.0
PAY_5,30000.0,-0.266200,1.133187,-2.0,8.0
PAY_6,30000.0,-0.291100,1.149988,-2.0,8.0
BILL_AMT1,30000.0,51223.330900,73635.860576,-165580.0,964511.0
BILL_AMT2,30000.0,49179.075167,71173.768783,-69777.0,983931.0


In [11]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                    rank                    
DefaultOnPaymentNextMonth 1       No  23364  77.88
                          2      Yes   6636  22.12
EDUCATION                 1        2  14030  46.77
                          2        1  10585  35.28
                          3        3   4917  16.39
                          4        5    280   0.93
                          5        4    123   0.41
MARRIAGE                  1        2  15964  53.21
                          2        1  13659  45.53
                          3        3    323   1.08
                          4        0     54   0.18
SEX                       1        2  18112  60.37
                          2        1  11888  39.63

In [12]:
# Target Distribution
target_df

,count,pct
DefaultOnPaymentNextMonth,,
No,23364,77.88
Yes,6636,22.12


## Task Curation

In [13]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=3, n_splits=3, test_size=None


In [14]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [15]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to credit_card_clients_default/019d5a3b-bdd6-77da-b7ac-8e0530344f06
019d5a3b-bdd6-77da-b7ac-8e0530344f06
83833eecb837ab7881750ab26c9cc34efe940dbb2eaebdc4e9e4b4998ee5c36c
